# Brier-penalty experiment

This notebook runs a one-epoch LoRA fine-tuning experiment with an added Brier-score penalty.

It reuses the exact data splits and accepted teacher traces from the original experiment. It does **not** regenerate datasets or teacher traces.

## Expected directory layout

Run this notebook with the working directory set to the extracted `brier_experiment_package` folder:

```text
brier_experiment_package/
├── data/splits.json
├── traces/train_traces.json
├── predictions/stage0_test.json
├── predictions/stage5_test.json
├── metrics/final_table.csv
└── source/current_jev.ipynb
```

The notebook creates a separate `brier_outputs/` directory. It does not overwrite the original experiment's artifacts.

The base model `Qwen/Qwen3-1.7B` must be available in the Hugging Face cache or downloadable from Hugging Face. The notebook uses the fallback model only if loading the primary model fails, matching the original notebook's behavior.

## 1. Configuration

The Brier coefficient is set to 1.0 and training runs for one epoch, matching the saved run metadata. The original experiment used two epochs.

In [ ]:
import os, json, gc, time, math, random, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 2026
BASE_MODEL = "Qwen/Qwen3-1.7B"
FALLBACK_MODEL = "Qwen/Qwen3-0.6B"

MAX_REASON_TOKENS = 256
EPOCHS = 1
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
BATCH_SIZE, GRAD_ACCUM = 1, 8
MAX_SEQ_LEN = 1024
BRIER_WEIGHT = 1.0

# The notebook expects to be launched from the extracted package directory.
ROOT = Path.cwd()
if not (ROOT / "data" / "splits.json").exists():
    raise FileNotFoundError(
        "data/splits.json was not found. Set the notebook working directory "
        "to the extracted brier_experiment_package folder."
    )

OUT = ROOT / "brier_outputs"
for d in ["checkpoints", "trainer", "predictions", "metrics", "plots"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)

CKPT = OUT / "checkpoints" / "lora_adapter_brier"
TRAINER_OUT = OUT / "trainer"

# Prevent accidentally reusing a previous Brier adapter.
if (CKPT / "adapter_config.json").exists():
    raise FileExistsError(
        f"{CKPT} already contains an adapter. Move/rename brier_outputs "
        "or deliberately remove that adapter before starting a fresh run."
    )

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Working directory:", ROOT.resolve())
print("Output directory:", OUT.resolve())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No CUDA GPU is visible. Do not run this training job on a login node.")

## 2. Load the exact original splits and teacher traces

The split file contains the prepared examples, including deterministic option order and labels. We load it directly rather than re-running the Hugging Face dataset mapping and split allocation.

In [ ]:
TASKS = [
    "sentiment",
    "inference",
    "science questions",
    "general knowledge",
    "word math",
]

DESCRIPTIONS = {
    "sentiment": "Classify the sentiment of the review as positive or negative.",
    "inference": "Determine whether the hypothesis is entailed, contradicted, or neutral given the premise.",
    "science questions": "Answer the grade school science question.",
    "general knowledge": "Answer the multiple-choice exam question.",
    "word math": "Solve the math word problem.",
}

with open(ROOT / "data" / "splits.json", "r", encoding="utf-8") as f:
    DATA = json.load(f)

with open(ROOT / "traces" / "train_traces.json", "r", encoding="utf-8") as f:
    traces = json.load(f)

for task in TASKS:
    if task not in DATA:
        raise KeyError(f"Missing task in splits.json: {task}")
    print(task, {split: len(DATA[task][split]) for split in ["train", "val", "test"]})

train_examples = [x for task in TASKS[:-1] for x in DATA[task]["train"]]
val_examples = [x for task in TASKS for x in DATA[task]["val"]]
test_examples = [x for task in TASKS for x in DATA[task]["test"]]

train_ids = {x["id"] for x in train_examples}
val_ids = {x["id"] for x in val_examples}
test_ids = {x["id"] for x in test_examples}
trace_ids = {t["id"] for t in traces}

assert not (train_ids & val_ids), "Training/validation leakage detected"
assert not (train_ids & test_ids), "Training/test leakage detected"
assert trace_ids.issubset(train_ids), "A teacher trace does not belong to a seen-task training example"
assert not (trace_ids & (val_ids | test_ids)), "Teacher trace leakage detected"

print("\nTraining examples:", len(train_examples))
print("Accepted teacher traces:", len(traces))
print("Validation examples:", len(val_examples))
print("Test examples:", len(test_examples))
print("Word Math training examples:", len(DATA["word math"]["train"]), "(excluded from training)")

## 3. Load tokenizer and define the original prompt/model helpers

These functions preserve the original notebook's prompt structure and answer-token readout.

In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    MODEL_NAME = BASE_MODEL
except Exception as e:
    print("Primary model load failed; using fallback:", repr(e))
    MODEL_NAME = FALLBACK_MODEL
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

LETTERS = list("ABCDE")
LETTER_IDS = {}
for letter in LETTERS:
    ids = tokenizer.encode(" " + letter, add_special_tokens=False)
    if len(ids) != 1:
        ids = tokenizer.encode(letter, add_special_tokens=False)
    assert len(ids) == 1, f"{letter} does not map to one token: {ids}"
    LETTER_IDS[letter] = ids[0]

print("Model:", MODEL_NAME)
print("Readout token IDs:", LETTER_IDS)


def prompt_for(ex, reasoning=False):
    body = f"Task: {DESCRIPTIONS[ex['task']]}\nQuestion: {ex['question']}\nOptions:\n"
    body += "".join(f"{LETTERS[i]}. {opt}\n" for i, opt in enumerate(ex["options"]))
    body += "Answer:"
    if reasoning:
        body += "\nReasoning:"
    return body


def load_model(name=MODEL_NAME, adapter=None):
    model = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=True,
    )
    if adapter is not None:
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()
    return model


def restricted_probs(logits, n, temp=1.0):
    ids = torch.tensor(
        [LETTER_IDS[x] for x in LETTERS[:n]],
        device=logits.device,
    )
    p = torch.softmax(logits[ids].float() / temp, dim=-1).cpu().numpy()
    assert np.isclose(p.sum(), 1, atol=1e-5)
    return p

## 4. Prediction helpers

The prediction path matches the original experiment: `fast` reads the answer distribution immediately after `Answer:`, while `reasoning` generates a scratchpad and then reads the distribution after `Final answer:`.

In [ ]:
def get_logits_hidden(model, prompt):
    x = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**x, output_hidden_states=True, use_cache=True)
    return out.logits[0, -1], out.hidden_states[-1][0, -1], x, out.past_key_values


def infer_one(model, ex, reasoning=False, max_new=MAX_REASON_TOKENS):
    start = time.time()
    p = prompt_for(ex, reasoning=False)
    logits, h, inputs, prompt_past = get_logits_hidden(model, p)
    fast = restricted_probs(logits, len(ex["options"]))

    if not reasoning:
        return {
            "probs": fast.tolist(),
            "hidden": h.float().cpu().tolist(),
            "generated_tokens": 0,
            "seconds": time.time() - start,
        }

    rp = prompt_for(ex, reasoning=True)
    rinputs = tokenizer(rp, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(
            **rinputs,
            max_new_tokens=max_new,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    seq = gen[0]
    scratch = tokenizer.decode(
        seq[rinputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    suffix = "\nFinal answer:"
    all_text = rp + scratch + suffix
    enc = tokenizer(all_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        rout = model(**enc, output_hidden_states=True)

    probs = restricted_probs(rout.logits[0, -1], len(ex["options"]))
    return {
        "probs": probs.tolist(),
        "hidden": rout.hidden_states[-1][0, -1].float().cpu().tolist(),
        "generated_tokens": int(seq.shape[0] - rinputs["input_ids"].shape[1]),
        "seconds": time.time() - start,
        "scratch": scratch,
    }


def save_json_atomic(rows, path, max_retries=5):
    import os
    path = Path(path)

    for attempt in range(max_retries):
        tmp = path.with_name(path.name + f".{os.getpid()}.{attempt}.tmp")
        try:
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump(rows, f)
                f.flush()
                os.fsync(f.fileno())
            tmp.replace(path)
            return
        except OSError as e:
            print(f"Checkpoint attempt {attempt + 1} failed: {e}")
            try:
                tmp.unlink(missing_ok=True)
            except OSError:
                pass
            if attempt == max_retries - 1:
                raise
            time.sleep(2 * (attempt + 1))


def run_predictions(model, examples, path, modes=("fast", "reasoning")):
    path = Path(path)
    existing = json.load(open(path, encoding="utf-8")) if path.exists() else []
    done = {r["id"] for r in existing}
    out = list(existing)

    print(f"Resuming: {len(done)} / {len(examples)} predictions already saved")

    for i, ex in enumerate(examples):
        if ex["id"] in done:
            continue

        row = {
            "id": ex["id"],
            "task": ex["task"],
            "label": ex["label"],
            "n_options": len(ex["options"]),
        }
        for mode in modes:
            row[mode] = infer_one(model, ex, mode == "reasoning")
        out.append(row)
        done.add(ex["id"])

        if len(done) % 5 == 0 or len(done) == len(examples):
            save_json_atomic(out, path)
            print("Saved predictions", len(done), "/", len(examples))

    byid = {r["id"]: r for r in out}
    out = [byid[x["id"]] for x in examples if x["id"] in byid]
    save_json_atomic(out, path)
    print("Completed:", len(out), "/", len(examples))
    return out

## 5. Build the training dataset

This reproduces the original mixed dataset construction: each seen-task training example appears once as a direct-answer row, and each accepted teacher trace adds a second reasoning-supervised row for its matching example.

In [ ]:
trace_by_id = {t["id"]: t for t in traces}
rows = [{"ex": x} for x in train_examples]
rows += [
    {"ex": next(x for x in train_examples if x["id"] == t["id"]), "trace": t}
    for t in traces
]
random.Random(SEED).shuffle(rows)

class MixDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        ex = r["ex"]
        prompt = prompt_for(ex)

        if r.get("trace"):
            text = prompt + "\nReasoning: " + r["trace"]["reasoning"] + "\nFinal answer:"
        else:
            text = prompt

        tok = tokenizer(
            text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
        )
        ids = tok["input_ids"][0]
        labels = ids.clone()

        if r.get("trace"):
            pids = tokenizer(
                prompt + "\nReasoning:",
                add_special_tokens=False,
            )["input_ids"]
            labels[:min(len(pids), len(labels))] = -100
        else:
            labels[:] = -100

        return {
            "input_ids": ids,
            "attention_mask": tok["attention_mask"][0],
            "labels": labels,
            "readout_pos": len(ids) - 1,
            "n_options": len(ex["options"]),
            "answer": ex["label"],
        }


class Collator:
    def __call__(self, items):
        # Right padding, preserving each unpadded readout position.
        mx = max(len(z["input_ids"]) for z in items)
        ids, masks, labs = [], [], []

        for z in items:
            n = mx - len(z["input_ids"])
            ids.append(torch.cat([
                z["input_ids"],
                torch.full((n,), tokenizer.pad_token_id, dtype=torch.long),
            ]))
            masks.append(torch.cat([
                z["attention_mask"],
                torch.zeros(n, dtype=torch.long),
            ]))
            labs.append(torch.cat([
                z["labels"],
                torch.full((n,), -100, dtype=torch.long),
            ]))

        return {
            "input_ids": torch.stack(ids),
            "attention_mask": torch.stack(masks),
            "labels": torch.stack(labs),
            "readout_pos": torch.tensor([z["readout_pos"] for z in items]),
            "n_options": torch.tensor([z["n_options"] for z in items]),
            "answer": torch.tensor([z["answer"] for z in items]),
        }


train_ds = MixDataset(rows)
print("Training rows:", len(train_ds))
print("Direct rows:", len(train_examples))
print("Reasoning-trace rows:", len(traces))

## 6. Initialize a fresh LoRA adapter and train with the Brier penalty

The answer probabilities are calculated only over the valid answer options. Invalid option slots are masked before softmax. The Brier term is the mean, across examples in a batch, of the sum of squared differences between the predicted option probabilities and the one-hot correct-answer vector.

In [ ]:
base = load_model()

linear_names = set()
for name, module in base.named_modules():
    if isinstance(module, torch.nn.Linear):
        leaf = name.split(".")[-1]
        if any(k in leaf.lower() for k in [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ]):
            linear_names.add(leaf)

assert linear_names, "No transformer projection layers discovered"
print("LoRA targets:", sorted(linear_names))

peft_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=sorted(linear_names),
    task_type="CAUSAL_LM",
)
model = get_peft_model(base, peft_cfg)
model.print_trainable_parameters()


class BrierPenaltyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        pos = inputs.pop("readout_pos")
        nopts = inputs.pop("n_options")
        answer = inputs.pop("answer")

        has_lm = (labels != -100).any().item()
        out = model(
            **inputs,
            labels=labels if has_lm else None,
        )

        option_ids = torch.tensor(
            [LETTER_IDS[x] for x in LETTERS[:int(nopts.max())]],
            device=out.logits.device,
        )

        z = out.logits[
            torch.arange(len(pos), device=out.logits.device),
            pos.to(out.logits.device),
        ][:, option_ids].float()

        option_mask = (
            torch.arange(z.shape[1], device=z.device)[None, :]
            >= nopts.to(z.device)[:, None]
        )
        z = z.masked_fill(option_mask, -1e4)

        answer = answer.to(z.device)
        answer_loss = torch.nn.functional.cross_entropy(z, answer)

        probs = torch.softmax(z, dim=-1)
        targets = torch.nn.functional.one_hot(
            answer,
            num_classes=z.shape[1],
        ).float()

        # Sum across options, then average across examples.
        brier_loss = torch.sum((probs - targets) ** 2, dim=-1).mean()

        lm_loss = out.loss if has_lm else z.sum() * 0.0
        loss = lm_loss + answer_loss + BRIER_WEIGHT * brier_loss

        if return_outputs:
            out.brier_loss = brier_loss.detach()
            out.answer_loss = answer_loss.detach()
            return loss, out
        return loss


args = TrainingArguments(
    output_dir=str(TRAINER_OUT),
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    fp16=(DEVICE == "cuda"),
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    max_grad_norm=1.0,
    remove_unused_columns=False,
    seed=SEED,
)

trainer = BrierPenaltyTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=Collator(),
)

print("Starting Brier-penalty training...")
train_result = trainer.train()
print(train_result)

model.save_pretrained(CKPT)
tokenizer.save_pretrained(CKPT)
print("Saved Brier adapter to:", CKPT)

del trainer, model, base
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 7. Calibration and evaluation helpers

Temperature fitting uses only the four seen tasks in the validation split, matching the original experiment. Word Math validation is not used to fit temperatures.

In [ ]:
def nll(probs, labels):
    return float(-np.mean([
        np.log(max(p[y], 1e-12))
        for p, y in zip(probs, labels)
    ]))


def fit_temperature(rows, mode):
    ps = np.array([r[mode]["probs"] for r in rows], dtype=object)
    ys = [r["label"] for r in rows]

    def loss(logt):
        t = np.exp(logt)
        scaled = [
            np.asarray(p) ** (1 / t) / np.sum(np.asarray(p) ** (1 / t))
            for p in ps
        ]
        return nll(scaled, ys)

    from scipy.optimize import minimize_scalar
    return float(np.exp(minimize_scalar(
        loss,
        bounds=(-3, 3),
        method="bounded",
    ).x))


def calib_metrics(probs, labels, bins=10):
    conf = np.array([max(p) for p in probs])
    pred = np.array([int(np.argmax(p)) for p in probs])
    ok = pred == np.array(labels)

    ece = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        hi = lo + 1 / bins
        ix = (conf >= lo) & (conf < (hi if hi < 1 else hi + 1e-8))
        if ix.any():
            ece += ix.mean() * abs(ok[ix].mean() - conf[ix].mean())

    width = max(map(len, probs))
    one = np.zeros((len(labels), width))
    pp = np.zeros_like(one)
    for i, y in enumerate(labels):
        one[i, y] = 1
    for i, p in enumerate(probs):
        pp[i, :len(p)] = p

    return {
        "accuracy": float(ok.mean()),
        "ece": float(ece),
        "nll": nll(probs, labels),
        "brier": float(np.mean(np.sum((pp - one) ** 2, axis=1))),
    }


def calibrated(rows, temps):
    for r in rows:
        for mode in ["fast", "reasoning"]:
            p = np.array(r[mode]["probs"], dtype=float)
            p = p ** (1 / temps[mode])
            r[mode]["calibrated_probs"] = (p / p.sum()).tolist()
    return rows


def metric_for(rows, mode, prob_key="calibrated_probs"):
    probs = [r[mode][prob_key] for r in rows]
    labels = [r["label"] for r in rows]
    return calib_metrics(probs, labels)


def eval_group(rows):
    result = {"n": len(rows)}
    for mode in ["fast", "reasoning"]:
        result[mode] = metric_for(rows, mode)
    confident = [
        r for r in rows
        if max(r["fast"]["calibrated_probs"]) >= 0.95
    ]
    result["confident_fast_count"] = len(confident)
    result["confident_wrong_fast_share"] = (
        float(np.mean([
            np.argmax(r["fast"]["calibrated_probs"]) != r["label"]
            for r in confident
        ]))
        if confident else None
    )
    return result

## 8. Generate validation predictions and fit temperatures

This runs both fast and reasoning predictions for all 750 validation examples. The temperature parameters are fitted on seen-task validation examples only.

In [ ]:
tuned = load_model(adapter=CKPT)

val_path = OUT / "predictions" / "brier_val.json"
val_rows = run_predictions(tuned, val_examples, val_path)

seen_val = [r for r in val_rows if r["task"] in TASKS[:-1]]
temps = {
    mode: fit_temperature(seen_val, mode)
    for mode in ["fast", "reasoning"]
}
with open(OUT / "metrics" / "temperatures.json", "w") as f:
    json.dump(temps, f, indent=2)

print("Brier-run temperatures:", temps)

val_rows = calibrated(val_rows, temps)
save_json_atomic(val_rows, OUT / "predictions" / "brier_val_calibrated.json")

del tuned
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 9. Generate test predictions and calculate metrics

This runs both prediction modes for all 1,250 test examples, then applies the temperatures fitted on seen-task validation data.

In [ ]:
tuned = load_model(adapter=CKPT)

test_path = OUT / "predictions" / "brier_test.json"
test_rows = run_predictions(tuned, test_examples, test_path)
test_rows = calibrated(test_rows, temps)
save_json_atomic(test_rows, OUT / "predictions" / "brier_test_calibrated.json")

del tuned
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

summary = {
    task: eval_group([r for r in test_rows if r["task"] == task])
    for task in TASKS
}
summary["overall"] = eval_group(test_rows)

with open(OUT / "metrics" / "brier_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

## 10. Compare against the completed original experiment

The comparison uses the original experiment's `metrics/final_table.csv` and the Brier run's newly calculated metrics. Both tables report calibrated fast and reasoning endpoints.

In [ ]:
brier_table_rows = []
for task in TASKS + ["overall"]:
    group = [r for r in test_rows if task == "overall" or r["task"] == task]
    row = {"task": task, "n": len(group)}
    for mode in ["fast", "reasoning"]:
        m = metric_for(group, mode)
        for key in ["accuracy", "ece", "nll", "brier"]:
            row[f"{mode}_{key}"] = m[key]
    row["confident_wrong_share"] = summary[task]["confident_wrong_fast_share"]
    brier_table_rows.append(row)

brier_df = pd.DataFrame(brier_table_rows)
brier_df.to_csv(OUT / "metrics" / "brier_final_table.csv", index=False)

original_path = ROOT / "metrics" / "final_table.csv"
if original_path.exists():
    original_df = pd.read_csv(original_path)
    comparison = original_df.merge(
        brier_df,
        on="task",
        suffixes=("_original", "_brier"),
        how="outer",
    )
    comparison.to_csv(OUT / "metrics" / "comparison_original_vs_brier.csv", index=False)
    display(comparison.round(4))
else:
    print("Original metrics/final_table.csv not found; showing Brier results only.")
    display(brier_df.round(4))

print("\nBrier results:")
display(brier_df.round(4))

## 11. Save run metadata

The metadata file records the model selected, training configuration, data counts, and package versions for reproducibility.

In [ ]:
import platform
import transformers, peft, datasets

metadata = {
    "base_model_requested": BASE_MODEL,
    "model_loaded": MODEL_NAME,
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "max_seq_len": MAX_SEQ_LEN,
    "brier_weight": BRIER_WEIGHT,
    "train_examples": len(train_examples),
    "accepted_teacher_traces": len(traces),
    "training_rows_including_trace_augments": len(rows),
    "validation_examples": len(val_examples),
    "test_examples": len(test_examples),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "datasets": datasets.__version__,
    "device": DEVICE,
    "temperatures": temps,
}

with open(OUT / "metrics" / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Brier experiment completed.")
print("Adapter:", CKPT)
print("Outputs:", OUT.resolve())